# Session 6 — Raster data with rasterio and xarray
*Séance 6 — Données matricielles avec rasterio et xarray*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/ML4EO_Bootcamp/blob/main/notebooks/06_raster_xarray.ipynb)

**Course:** Machine Learning for Earth Observation (45 h) · **Instructor:** Noé Carème Fouotsa Manfouo

**By the end of this session you will be able to:**

1. Describe a raster: grid, resolution, CRS, affine transform, bands, no-data.
2. Read, write and clip GeoTIFFs with rasterio.
3. Compute zonal statistics inside a polygon.
4. Handle multi-date image stacks with xarray.

**How to use this notebook:** read the lecture cells, run every code cell (`Shift + Enter`), then complete the cells marked
`# YOUR CODE HERE` in the lab section. Run the ✅ check cells to get instant feedback.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
import sys, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("ML4EO_Bootcamp").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/ML4EO_Bootcamp.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *"geopandas rasterio rioxarray".split()], check=True)
    ROOT = pathlib.Path("ML4EO_Bootcamp")
else:
    ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
sys.path.append(str(ROOT / "notebooks"))   # course helper module (ml4eo_data.py)


def check(condition, message="Check failed"):
    """Instant feedback: raises an error with a hint if the condition is False."""
    assert condition, message
    print("✅ Passed")


def check_equal(value, expected, message=None):
    assert value == expected, message or f"Expected {expected!r}, got {value!r}"
    print("✅ Passed")

## Part A — Lecture

A **raster** is a grid of pixels. Its georeferencing is the **CRS** plus an **affine transform** that maps pixel
(row, column) to map coordinates (x, y). We first create a small Sentinel-2-like GeoTIFF over our ward (10 m pixels, UTM 34S).

In [ ]:
import numpy as np

BANDS = ["B2", "B3", "B4", "B8", "B11"]                       # blue, green, red, NIR, SWIR1
WAVELENGTH_NM = [490, 560, 665, 842, 1610]
# typical surface reflectance (0-1) of four land-cover types for these bands
LIBRARY = {"water":      [0.06, 0.07, 0.05, 0.03, 0.01],
           "vegetation": [0.03, 0.07, 0.04, 0.40, 0.20],
           "bare soil":  [0.10, 0.14, 0.18, 0.25, 0.32],
           "built-up":   [0.12, 0.13, 0.14, 0.18, 0.30]}


def synthetic_scene(size=200, seed=0):
    """Return (cube, labels): a Sentinel-2-like reflectance cube (bands, rows, cols) and the true class of each pixel."""
    rng = np.random.default_rng(seed)
    y, x = np.mgrid[0:size, 0:size]
    labels = np.full((size, size), 2)                                      # bare soil background
    labels[(x - 0.3 * size) ** 2 + (y - 0.35 * size) ** 2 < (0.15 * size) ** 2] = 0   # a lake
    labels[(y > 0.55 * size) & (x < 0.6 * size)] = 1                                  # fields
    labels[(x > 0.68 * size) & (y > 0.2 * size) & (y < 0.8 * size)] = 3               # a town
    names = list(LIBRARY)
    mean = np.array([LIBRARY[n] for n in names])[labels]                   # rows, cols, bands
    cube = mean * rng.lognormal(0, 0.08, mean.shape)
    return np.moveaxis(cube, -1, 0).astype("float32"), labels

import geopandas as gpd, rasterio
from rasterio.transform import from_origin

ward = gpd.read_file(DATA / "SA_LEVEL4" / "ADM4_PCODE_ZA1011001.shp").to_crs(32734)
xmin, ymin, xmax, ymax = ward.total_bounds
cube, truth = synthetic_scene(size=200, seed=1)
transform = from_origin(xmin + 6000, ymax - 6000, 10, 10)   # 2 km x 2 km inside the ward

profile = {"driver": "GTiff", "height": 200, "width": 200, "count": len(BANDS), "dtype": "float32",
           "crs": "EPSG:32734", "transform": transform, "nodata": -9999, "compress": "deflate"}
with rasterio.open("s2_subset.tif", "w", **profile) as dst:
    dst.write(cube)
    dst.descriptions = tuple(BANDS)
print(transform)

In [ ]:
with rasterio.open("s2_subset.tif") as src:
    print(src.profile)
    print("Bounds:", src.bounds, "| resolution:", src.res)
    red, nir = src.read(3), src.read(4)
    print("Pixel (0, 0) centre in map coordinates:", src.xy(0, 0))

### Writing a derived product
Always copy the source profile and change only what differs (here one band).

In [ ]:
ndvi = (nir - red) / (nir + red)
with rasterio.open("s2_subset.tif") as src:
    out = src.profile | {"count": 1}
with rasterio.open("ndvi.tif", "w", **out) as dst:
    dst.write(ndvi.astype("float32"), 1)

### Clipping with a polygon

In [ ]:
from rasterio.mask import mask
from shapely.geometry import box

field = gpd.GeoSeries([box(xmin + 6200, ymax - 7900, xmin + 7000, ymax - 7200)], crs=32734)   # a field polygon
with rasterio.open("ndvi.tif") as src:
    clipped, clip_transform = mask(src, field.geometry, crop=True, nodata=-9999)
print(clipped.shape)

### Image time series with xarray
`xarray` labels the dimensions (`time`, `y`, `x`), which makes multi-date analysis readable.

In [ ]:
import pandas as pd, xarray as xr
import matplotlib.pyplot as plt

dates = pd.date_range("2021-01-15", periods=12, freq="MS") + pd.Timedelta(days=14)
season = 0.5 + 0.35 * np.cos((dates.month.to_numpy() - 1) / 12 * 2 * np.pi)   # southern-hemisphere summer green-up
rng = np.random.default_rng(0)
stack = np.where(truth == 1, season[:, None, None], 0.1) + rng.normal(0, 0.03, (12, 200, 200))
ndvi_ts = xr.DataArray(stack, dims=("time", "y", "x"),
                       coords={"time": dates, "y": ymax - 6000 - 5 - 10 * np.arange(200), "x": xmin + 6000 + 5 + 10 * np.arange(200)},
                       name="ndvi")
ndvi_ts.where(xr.DataArray(truth == 1, dims=("y", "x"))).mean(("y", "x")).plot(marker="o")
plt.title("Mean NDVI of fields"); plt.show()

## Part B — Lab

**Task 1 — Raster metadata.** Open `s2_subset.tif` and store its CRS EPSG code (`epsg`), pixel size in metres (`pixel_size`)
and area covered in hectares (`area_ha`).

In [ ]:
# YOUR CODE HERE
epsg = None
pixel_size = None
area_ha = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check_equal(epsg, 32734)
check_equal(pixel_size, 10.0)
check_equal(area_ha, 400.0)

**Task 2 — Zonal statistics.** Using the clipped array `clipped` (no-data = −9999), compute the mean NDVI of the field
(`field_mean`) and the share of its pixels with NDVI > 0.6 (`share_dense`).

In [ ]:
# YOUR CODE HERE
field_mean = None
share_dense = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(0 < field_mean < 1)
check(0 <= share_dense <= 1)
print(f"Mean NDVI {field_mean:.2f}, dense share {share_dense:.0%}")

**Task 3 — Seasonal composite.** From `ndvi_ts`, compute `summer` = the mean NDVI over December–February and `winter` =
the mean over June–August (2-D DataArrays). Then `amplitude = summer - winter`.

In [ ]:
# YOUR CODE HERE
summer = None
winter = None
amplitude = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(float(amplitude.values[truth == 1].mean()) > 0.4, "Fields should show a strong seasonal amplitude")
amplitude.plot(cmap="viridis"); plt.title("Seasonal NDVI amplitude"); plt.show()

### Optional — real Sentinel-2 data (needs internet)
Set `USE_REAL_DATA = True` to search the free Sentinel-2 Level-2A archive on AWS (Element 84 Earth Search STAC API) and read
one band of a cloud-optimized GeoTIFF directly, without downloading the whole scene.

In [ ]:
USE_REAL_DATA = False
if USE_REAL_DATA:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pystac-client"], check=True)
    from pystac_client import Client
    lon, lat = 18.36, -31.60
    items = Client.open("https://earth-search.aws.element84.com/v1").search(
        collections=["sentinel-2-l2a"], intersects={"type": "Point", "coordinates": [lon, lat]},
        datetime="2021-01-01/2021-02-28", query={"eo:cloud_cover": {"lt": 10}}, max_items=1).item_collection()
    item = items[0]
    print(item.id, item.properties["eo:cloud_cover"])
    with rasterio.open(item.assets["nir"].href) as src:
        print(src.profile["width"], "x", src.profile["height"], "pixels at", src.res)

## Summary

- A raster = grid + CRS + affine transform (+ no-data).
- Copy and update profiles when writing products; clip with polygons for zonal statistics.
- xarray makes multi-date stacks readable (`sel`, `mean('time')`, `.dt.month`).

**To go further:**

- rasterio documentation: https://rasterio.readthedocs.io
- xarray tutorial: https://tutorial.xarray.dev